In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

In [2]:
load_dotenv()

# Load existing vector store
embeddings = OpenAIEmbeddings()

vectorstore = Chroma(
    persist_directory='../../chroma_db',
    embedding_function=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

print('Vector store loaded and retriever ready.')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store loaded and retriever ready.


Step 2: Define the Query Transformer

In [3]:
# Create LLM for query rewriting (deterministic)
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Prompt to instruct the model to rewrite the query
transform_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Rewrite the user question to make it more specific and likely to retrieve relevant information from a knowledge base about agriculture and health in Nigeria.'),
    ('human', '{question}')
])

# Build the transformer chain: prompt -> llm -> string
query_transformer = transform_prompt | llm | StrOutputParser()

print('Query transformer ready.')

Query transformer ready.


Step 3: Test Query Transformation with a Real Query

In [4]:
# Define the original problematic query
raw_query = 'What are Nigeria communicable and infectious diseases?'

# Transform the query
transform_query = query_transformer.invoke({'question': raw_query})
print(f'Original query: {raw_query}')
print(f'Transformed query: {transform_query}')

# Retrieve with original query
raw_docs = retriever.invoke(raw_query)
print('\nWithout transformation:')
for doc in raw_docs:
    print('-', doc.page_content)
    print('  Source:', doc.metadata.get('source', 'unknown'))

Original query: What are Nigeria communicable and infectious diseases?
Transformed query: What are the most prevalent communicable and infectious diseases in Nigeria, and how do they impact agricultural practices and public health in the country?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Without transformation:
- officia veritatis tenetur vero qui itaque
sint non ratione
sed et ut asperiores iusto eos molestiae nostrum
veritatis quibusdam et nemo iusto saepe
  Source: C:\Users\USER\rag_course\04_data_ingestion_document_processing\data\ecommerce.json
- officia veritatis tenetur vero qui itaque
sint non ratione
sed et ut asperiores iusto eos molestiae nostrum
veritatis quibusdam et nemo iusto saepe
  Source: C:\Users\USER\rag_course\04_data_ingestion_document_processing\data\ecommerce.json
- officia veritatis tenetur vero qui itaque
sint non ratione
sed et ut asperiores iusto eos molestiae nostrum
veritatis quibusdam et nemo iusto saepe
  Source: C:\Users\USER\rag_course\04_data_ingestion_document_processing\data\ecommerce.json
- earum voluptatem facere provident blanditiis velit laboriosam
pariatur accusamus odio saepe
cumque dolor qui a dicta ab doloribus consequatur omnis
corporis cupiditate eaque assumenda ad nesciunt
  Source: C:\Users\USER\rag_course\04_data_inges

In [5]:
import os
print('Current working directory:', os.getcwd())

Current working directory: c:\Users\USER\rag_course\07_advanced_retrieval\notebooks


 Retrieve with Transformed Query

In [6]:
# Original query
raw_query = 'What are Nigeria communicable and infectious diseases?'

# Transform the query (if not already transformed)
transformed_query = query_transformer.invoke({'question': raw_query})
print('Transformed query:', transformed_query)

# Retrieve using transformed query
transformed_docs = retriever.invoke(transform_query)

print('\nRetrieved chunks with transformed query:')

for i, doc in enumerate(transformed_docs, start=1):
    print(f'{i}. {doc.page_content}')
    print(f'    Source: {doc.metadata.get("source", "unknown")}')

Transformed query: What are the most prevalent communicable and infectious diseases in Nigeria, and how do they impact agricultural practices and public health in the country?

Retrieved chunks with transformed query:
1. officia veritatis tenetur vero qui itaque
sint non ratione
sed et ut asperiores iusto eos molestiae nostrum
veritatis quibusdam et nemo iusto saepe
    Source: C:\Users\USER\rag_course\04_data_ingestion_document_processing\data\ecommerce.json
2. officia veritatis tenetur vero qui itaque
sint non ratione
sed et ut asperiores iusto eos molestiae nostrum
veritatis quibusdam et nemo iusto saepe
    Source: C:\Users\USER\rag_course\04_data_ingestion_document_processing\data\ecommerce.json
3. officia veritatis tenetur vero qui itaque
sint non ratione
sed et ut asperiores iusto eos molestiae nostrum
veritatis quibusdam et nemo iusto saepe
    Source: C:\Users\USER\rag_course\04_data_ingestion_document_processing\data\ecommerce.json
4. earum voluptatem facere provident blandit

In [7]:
# LLM for query rewriting
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

transform_prompt = ChatPromptTemplate.from_messages([
    ('system'
     'You are an expert query rewriter for a RAG system. '
     'The knowledge base contains documents about agriculture, public health, and e-commerce in Nigeria. '
     'Rewrite the user question to be more specific and focused on the intended domain. '
     'If the question is about health, use health-related terms only. '
     'If about agriculture, use agriculture terms only. '
     'Do NOT introduce topics outside the original domain.'
     ),
    ('human', '{question}')
])

# Build the transformer chain
query_transformer2 = transform_prompt | llm | StrOutputParser()

print('Improved query transformer ready.')

Improved query transformer ready.


In [8]:
raw_query = 'What are Nigeria communicable and infectious diseases?'

# Transform

transform_query1 = query_transformer2.invoke({'question': raw_query})

print(f'Original query: {raw_query}')
print(f'Transformed query: {transform_query}')

Original query: What are Nigeria communicable and infectious diseases?
Transformed query: What are the most prevalent communicable and infectious diseases in Nigeria, and how do they impact agricultural practices and public health in the country?


**What is Self-Query Retriever?**

It parses the user question into two parts:

* **Semantic query** – the actual question text.
* **Metadata filter** – conditions like doc_type == 'pdf' or source contains 'health'.

It then applies that filter automatically during retrieval.

This eliminates the need for manual filtering and keeps the query focused.

In [9]:
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo
from langchain_openai import ChatOpenAI



Step 2: Define Metadata Field Information

In [10]:
# Describe the metadata fields in vector store

metadata_field_info = [
    AttributeInfo(
        name='source',
        description='The file path of the document. Contains keywords like "nigeria_health" or "crop_disease" or "ecommerce".',
        type='string',
    ),
    AttributeInfo(
        name='doc_type',
        description='The type of document: pdf, html, txt, csv, json',
        type='string',
    ),
    AttributeInfo(
        name='language',
        description='Language of the document, e.g., English',
        type='string',
    ),
]

Step 3: Define the Document Content Description

In [11]:
document_content_description = 'Documents about agriculture, public health, and e-commerce in Nigeria'

Step 4: Create the Self-Query Retriever

Step 4: Create the Self-Query Retriever

In [12]:
# Use the same LLM for query parsing

llm_1 = ChatOpenAI(model='gpt-4o', temperature=0)

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm_1,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verbose=True     #To see the generated filter
)

Step 5: Test with Query

In [13]:
raw_query = 'What are Nigeria communicable and infectious diseases?'

# Retrieve using self-query retriever
docs_1 = self_query_retriever.invoke(raw_query)

for i, d in enumerate(docs_1, start=1):
    print(f'{i}, {doc.page_content}')
    print(f'     Source: {d.metadata.get("source", "unknown")}')
    print(f'     Type: {d.metadata.get("doc_type", "unknown")}')

1, earum voluptatem facere provident blanditiis velit laboriosam
pariatur accusamus odio saepe
cumque dolor qui a dicta ab doloribus consequatur omnis
corporis cupiditate eaque assumenda ad nesciunt
     Source: 04_data_ingestion_document_processing\data\nigeria_health_diseases_and_prevention.pdf
     Type: pdf
2, earum voluptatem facere provident blanditiis velit laboriosam
pariatur accusamus odio saepe
cumque dolor qui a dicta ab doloribus consequatur omnis
corporis cupiditate eaque assumenda ad nesciunt
     Source: 04_data_ingestion_document_processing\data\nigeria_health_diseases_and_prevention.pdf
     Type: pdf
3, earum voluptatem facere provident blanditiis velit laboriosam
pariatur accusamus odio saepe
cumque dolor qui a dicta ab doloribus consequatur omnis
corporis cupiditate eaque assumenda ad nesciunt
     Source: 04_data_ingestion_document_processing\data\nigeria_health_diseases_and_prevention.pdf
     Type: pdf
4, earum voluptatem facere provident blanditiis velit laborio